## Reinforcement Learning: Basic Concepts

Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with an environment. The agent receives feedback in the form of rewards or penalties based on its actions, and its goal is to maximize the cumulative reward over time.

### Environment: Treasure Corridor

The agent moves in a one-dimensional corridor with five states: 0, 1, 2, 3, 4.

- State 0 is a bad terminal state with reward -1 when entered.
- State 4 is a good terminal state with reward +1 when entered.
- States 1, 2, 3 are non-terminal.
- Every non-terminal move has a small step cost -0.02.
- Actions are left and right.
- The action succeeds with probability 0.80; with probability 0.20, the agent moves in the opposite direction.



In [1]:
from collections import Counter, defaultdict
import random

GAMMA = 0.90
STATES = (0, 1, 2, 3, 4)
TERMINALS = {0, 4}
ACTIONS = ("left", "right")


def is_terminal(state):
    return state in TERMINALS


print("States:", STATES)
print("Terminal states:", TERMINALS)
print("Actions:", ACTIONS)

States: (0, 1, 2, 3, 4)
Terminal states: {0, 4}
Actions: ('left', 'right')


#### Transition model

Transition model is a function that takes a state and an action and returns a list of possible next states, their probabilities, and the rewards associated with those transitions.

It must return a list of tuples: `(probability, next_state, reward)`.

For example, from state 2 with action right, the agent should usually move to 3, but sometimes move to 1.

In [2]:
def transition_distribution(state, action):
    """Return possible stochastic outcomes for one state-action pair."""
    if state in TERMINALS:
        return [(1.0, state, 0.0)]

    # convert the action into an intended move.
    # Hint: left is -1, right is +1.
    intended_move = -1 if action == "left" else 1

    # define the opposite move.
    opposite_move = -intended_move

    outcomes = []
    for probability, move in [(0.80, intended_move), (0.20, opposite_move)]:
        # compute the next state, clipped to the corridor boundaries.
        next_state = max(0, min(4, state + move))

        # assign the reward for entering next_state.
        # Entering 0 gives -1.0; entering 4 gives +1.0; otherwise -0.02.
        reward = -1.0 if next_state == 0 else (1.0 if next_state == 4 else -0.02)

        outcomes.append((probability, next_state, reward))

    return outcomes



### Sampling a transition and running an episode

The helper step samples one outcome from your transition distribution. Then you will implement two policies and use them to generate episodes.

In [3]:
def step(state, action, rng):
    sample = rng.random()
    total = 0.0
    for probability, next_state, reward in transition_distribution(state, action):
        total += probability
        if sample <= total:
            return next_state, reward
    probability, next_state, reward = transition_distribution(state, action)[-1]
    return next_state, reward


def policy_go_right(state):
    # return the action that always moves toward the good terminal state.
    return "right"
    raise NotImplementedError("TODO: implement policy_go_right")


def policy_random(state, rng):
    # return a random action from ACTIONS.
    return rng.choice(ACTIONS)
    raise NotImplementedError("TODO: implement policy_random")


def run_episode(policy, start_state=2, seed=0, max_steps=20):
    rng = random.Random(seed)
    state = start_state
    episode = []

    for _ in range(max_steps):
        if is_terminal(state):
            break
        try:
            action = policy(state, rng)
        except TypeError:
            action = policy(state)
        next_state, reward = step(state, action, rng)
        episode.append((state, action, reward, next_state))
        state = next_state

    return episode


def print_episode(episode):
    for t, (state, action, reward, next_state) in enumerate(episode):
        print(f"t={t:02d}: s={state}, a={action:>5}, r={reward:+.2f}, s_next={next_state}")

### Discounted return

Discounted return is the total reward an agent can expect to accumulate over the future, starting from a given state. It is calculated by summing the rewards received at each time step, discounted by a factor gamma (0 < gamma < 1) that represents the importance of future rewards.

For rewards r1, r2, r3, ..., the discounted return is:

$$G = r1 + gamma*r2 + gamma^2*r3 + ...
$$

In [4]:
def discounted_return(rewards, gamma=GAMMA):
    # compute the discounted sum of rewards.
    total = 0.0
    discount = 1.0
    for reward in rewards:
        total += discount * reward
        discount *= gamma
    return total


# Try it on one episode.
episode = run_episode(policy_go_right, seed=7)
rewards = [reward for _, _, reward, _ in episode]
print("Rewards:", rewards)
print("Discounted return:", discounted_return(rewards))

Rewards: [-0.02, 1.0]
Discounted return: 0.88


### Monte Carlo value estimation

A value function estimates the expected return from each state under a policy. In this exercise, use complete episodes to estimate values by averaging observed returns-to-go.

Monte Carlo methods are model-free, meaning they do not require knowledge of the transition model. They rely on sampling episodes and using the observed returns to update value estimates.

In [5]:
def returns_to_go(episode, gamma=GAMMA):
    """Return a list of (state, return_from_that_state) pairs."""
    # walk backward through the episode and compute the return from each visited state.
    result = []
    running_return = 0.0
    for state, action, reward, next_state in reversed(episode):
        running_return = reward + gamma * running_return
        result.append((state, running_return))
    return list(reversed(result))


def monte_carlo_values(policy, episodes=200, seed=12):
    rng = random.Random(seed)
    returns_by_state = defaultdict(list)

    for episode_index in range(episodes):
        start_state = rng.choice([1, 2, 3])
        episode = run_episode(policy, start_state=start_state, seed=seed + episode_index)
        for state, value_sample in returns_to_go(episode):
            returns_by_state[state].append(value_sample)

    return {
        state: sum(samples) / len(samples)
        for state, samples in returns_by_state.items()
        if samples
    }


mc_values = monte_carlo_values(policy_go_right, episodes=50)
print(dict(sorted((state, round(value, 3)) for state, value in mc_values.items())))

{1: 0.391, 2: 0.705, 3: 0.939}


### Bellman policy evaluation

Bellman evaluation is a method for computing the value function for a given policy. It uses the Bellman equation to iteratively update the value estimates based on the expected returns from each state.

For a fixed policy, Bellman evaluation repeatedly applies this backup:

$$V(s) = \sum_{s'} P(s' | s, \pi(s)) \left[ R(s, \pi(s), s') + \gamma V(s') \right] $$


In [6]:
def bellman_backup_for_policy(state, policy, values, gamma=GAMMA):
    if is_terminal(state):
        return 0.0

    action = policy(state)
    expected_value = 0.0

    # use transition_distribution to compute the Bellman expected value.
    for probability, next_state, reward in transition_distribution(state, action):
        expected_value += probability * (reward + gamma * values[next_state])

    return expected_value


def evaluate_policy_with_bellman(policy, iterations=50):
    values = {state: 0.0 for state in STATES}
    for _ in range(iterations):
        new_values = values.copy()
        for state in STATES:
            new_values[state] = bellman_backup_for_policy(state, policy, values)
        values = new_values
    return values


bellman_values = evaluate_policy_with_bellman(policy_go_right)
print({state: round(value, 3) for state, value in bellman_values.items()})

{0: 0.0, 1: 0.284, 2: 0.694, 3: 0.921, 4: 0.0}


### Temporal-difference update

TD(0) updates the value of the current state after observing one transition:

$$V(s) \leftarrow V(s) + \alpha \left[ \text{reward} + \gamma V(s_{\text{next}}) - V(s) \right] $$


In [7]:
def td_update(current_value, reward, next_value, alpha=0.10, gamma=GAMMA):
    # compute one TD(0) update and return the new current value.
    return current_value + alpha * (reward + gamma * next_value - current_value)


def td_learning(policy, episodes=300, alpha=0.10, seed=21):
    rng = random.Random(seed)
    values = defaultdict(float)

    for episode_index in range(episodes):
        state = rng.choice([1, 2, 3])
        for _ in range(20):
            if is_terminal(state):
                break
            action = policy(state)
            next_state, reward = step(state, action, rng)
            values[state] = td_update(values[state], reward, values[next_state], alpha=alpha)
            state = next_state

    return values


td_values = td_learning(policy_go_right)
print({state: round(td_values[state], 3) for state in STATES})

{0: 0.0, 1: -0.105, 2: 0.564, 3: 0.882, 4: 0.0}


### Exploration with epsilon-greedy action selection

An agent that always exploits current estimates may stop exploring too early. Implement epsilon-greedy action selection for action utilities Q(s, a).



In [8]:
def epsilon_greedy_action(q_values, state, epsilon, rng):
    # with probability epsilon, choose a random action.
    # Otherwise choose the action with the largest Q-value.
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)
    return max(ACTIONS, key=lambda action: q_values[(state, action)])

    raise NotImplementedError("TODO: implement epsilon_greedy_action")


q_values = defaultdict(float)
q_values[(2, "left")] = -0.20
q_values[(2, "right")] = 0.50

rng = random.Random(99)
counts = Counter(epsilon_greedy_action(q_values, 2, epsilon=0.25, rng=rng) for _ in range(1000))
print(counts)

Counter({'right': 847, 'left': 153})
